# Data preparation

The aim of this notebook is to use the polygon API to extract all the data I require for this project and rewrite it in a way where it is cleaner. All I need is open and close prices to see its daily flucuatations.

## Imports and pre-requistites

In [ ]:
import pandas as pd
import polygon 
from dotenv import load_dotenv

import os
from pathlib import Path

In [ ]:
#Prerequisites
load_dotenv("../.env")
POLYGON_KEY = os.getenv("POLYGON_KEY")

## Processing the constituents

In [ ]:
smp500_constituents= pd.read_csv("../data/raw/sp500_historical_constituents.csv")

In [ ]:
smp500_constituents.head()

An important note here is that it is not daily therefore we will have to compensate for that later within the project.

In [ ]:
# Remove the rows which are outside our desired date range (2004-01-01 to 2024-12-31) 
smp500_constituents = smp500_constituents[(smp500_constituents['date'] >= '2004-01-01') & (smp500_constituents['date'] <= '2024-12-31')]

In [ ]:
smp500_constituents.head()

In [ ]:
#Check for missing values
smp500_constituents.isnull().sum()

Describing the data would be slightly impractical as it is strings

In [ ]:
smp500_constituents["tickers"] = smp500_constituents["tickers"].str.split(",")

In [ ]:
#Extract all unique tickers from the 'tickers' column and store them in a new array called "unique_tickers"
unique_tickers = smp500_constituents['tickers'].explode().unique()
unique_tickers

## Data collection

Here, we are going to set up the raw stock market data pulled from polygon API. 
Yahoo finance can be used as well but I have a polygon API key so I will utilise it.

In [ ]:
DATA_DIR = Path("../data/raw/polygon_initial")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
client = polygon.StocksClient(POLYGON_KEY, True)

In [ ]:
async def fetch_ticker(ticker):
    try:
        bars = await client.get_aggregate_bars(
            ticker,
            '2004-01-01',
            '2024-12-31',
            full_range=True
        )

        df = pd.DataFrame(bars)

        df.to_csv(DATA_DIR / f"{ticker}.csv", index=False)

    except Exception as e:
        print(f"{ticker}: {e}")

for ticker in unique_tickers:
    await fetch_ticker(ticker)

## Data preprocessing (Reformatting and filtering)

In [ ]:
# Rewrite the files into the structure of date,open,close and filter
STOCK_DATA_REFORMATTED = Path("../data/interim/reformatted_date_open_close")
STOCK_DATA_REFORMATTED.mkdir(parents=True, exist_ok=True)
for file in DATA_DIR.glob("*.csv"):
    try: #This is incase there are any problems (e.g. missing data)
        df = pd.read_csv(file)
        df = df[[ 'o', 'c' , "date"]] #Extracting only the columns we need
        df = df.iloc[:, [2, 0, 1]]
        df.columns = ['date', 'open', 'close']
        df = df[['date', 'open', 'close']]
        #Save the file into stock_data_reformatted
        df.to_csv(STOCK_DATA_REFORMATTED / f"{file.stem}.csv", index=False)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        #Delete said file
        os.remove(file)
        continue

All of the data starts from 2003-12-9 so I will filter from 2004 in which the stock market data starts from 2004-01-07 (YYYY-MM-DD)

In [ ]:
#Filter the files in stock_data_filtered_by_date to only include data from 2004-01-01 to 2024-12-31
INPUT_DIR = Path("../data/interim/reformatted_date_open_close")
STOCK_DATA_FILTERED_BY_DATE = Path("../data/interim/filtered_by_date")
STOCK_DATA_FILTERED_BY_DATE.mkdir(parents=True, exist_ok=True)
for file in INPUT_DIR.glob("*.csv"):
    try:
        df = pd.read_csv(file)
        df = df[(df['date'] >= '2004-01-01') & (df['date'] <= '2024-12-31')]
        df.to_csv(STOCK_DATA_FILTERED_BY_DATE / f"{file.stem}.csv", index=False)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        #Delete said file
        os.remove(file)
        continue

Looking at the data, the dates start from 2004-01-07 (YYYY-MM-DD)

Now we have to filter it so the companies only have the stock data from the time that they were in the S&P500

In [ ]:

# Rereading the data to ensure that the data is in the correct format and that there are no missing values
historical_constituents = pd.read_csv(
    "../data/raw/sp500_historical_constituents.csv",
    parse_dates=["date"]
).sort_values("date")

historical_constituents["tickers"] = historical_constituents["tickers"].str.split(",")


long = (
    historical_constituents
    .explode("tickers")
)

long["tickers"] = long["tickers"].str.strip()

In [ ]:
snapshot_dates = historical_constituents["date"].sort_values().unique()

membership = {}

for ticker, grp in long.groupby("tickers"):
    dates = sorted(grp["date"])

    intervals = []

    for d in dates:
        pos = snapshot_dates.searchsorted(d)

        if pos + 1 < len(snapshot_dates):
            end = snapshot_dates[pos + 1]
        else:
            end = pd.Timestamp.max

        intervals.append((d, end))

    membership[ticker] = intervals

In [ ]:
OUT_DIR= Path("../data/interim/filtered_sp500_membership")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for file in STOCK_DATA_FILTERED_BY_DATE.glob("*.csv"):

    ticker = file.stem.upper()

    if ticker not in membership:
        continue

    df = pd.read_csv(file, parse_dates=["date"])

    keep = pd.Series(False, index=df.index)

    for start, end in membership[ticker]:
        keep |= (df["date"] >= start) & (df["date"] < end)

    filtered = df[keep]

    filtered.to_csv(OUT_DIR / file.name, index=False)